# Day 45: SGLang: Structured Output Latency Benchmark

**Layer:** Implementation


## Concept Overview

SGLang's constrained decoding produces guaranteed-valid JSON/regex outputs by masking logits via FSM at each decode step. The overhead is <5% for simple schemas and eliminates the need for retry logic.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. SGLang Deployment and Structured Output Latency

SGLang extends vLLM with RadixAttention and constrained decoding. Deploy and benchmark with JSON-formatted outputs.


In [ ]:
print('SGLang deployment and structured output benchmark:')
print()
print('Launch command:')
print('  python -m sglang.launch_server \\')
print('    --model-path meta-llama/Llama-3.1-8B-Instruct \\')
print('    --port 30000 --tp 1')
print()
print('Structured output request example:')
structured_request = {
    'model': 'llama-3.1-8b-instruct',
    'messages': [{'role': 'user', 'content': 'Extract name and age from: John is 25.'}],
    'response_format': {
        'type': 'json_schema',
        'json_schema': {
            'name': 'extraction',
            'schema': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'age': {'type': 'integer'}}}
        }
    }
}
import json
print(json.dumps(structured_request, indent=2))


## 2. Structured Output Latency Analysis

Constrained decoding adds overhead at each token: the FSM transition must be computed before sampling. Measure this overhead.


In [ ]:
import time, numpy as np

def simulate_constrained_decoding(output_tokens, fsm_overhead_us=50):
    """Simulate decode with FSM overhead per token."""
    decode_time_ms = 20.0  # base TPOT
    times = []
    for _ in range(output_tokens):
        # Base decode
        t = decode_time_ms + np.random.normal(0, 1)
        # FSM overhead
        t += fsm_overhead_us / 1000
        times.append(t)
    return times

unconstrained = simulate_constrained_decoding(100, fsm_overhead_us=0)
constrained = simulate_constrained_decoding(100, fsm_overhead_us=100)

print('Structured output overhead:')
print(f'  Unconstrained TPOT: {np.mean(unconstrained):.2f} ms')
print(f'  Constrained TPOT:   {np.mean(constrained):.2f} ms')
print(f'  Overhead: {(np.mean(constrained)/np.mean(unconstrained)-1)*100:.1f}%')
print()
print('Benefit: zero retries for malformed JSON')
print('Real SGLang overhead: typically <5% for simple JSON schemas')


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- SGLang's constrained decoding produces guaranteed-valid JSON/regex outputs by masking logits via FSM at each decode step.
- Constrained decoding overhead is proportional to FSM complexity: a simple JSON schema with 2 fields adds <1ms per token; a complex nested schema may add 5-10ms..
- Day 45 implementation complete.
